# InvestingLab · demo del taller B1-T1

Recorrido completo por la librería: descarga simultánea de varias series, objetos validados con
estadísticos automáticos, cartera en euros, simulación de Monte Carlo e informe con gráficos.

**Datos.** El `Toolkit` usa los proveedores configurados en tu entorno (`INVESTINGLAB_*`). Hasta
que se conecten los remotos (fase F2b), apunta `INVESTINGLAB_LOCAL_DATA_DIR` a una carpeta con
tus ficheros (`SAN.MC.csv`, …) y un `instruments.toml` que describa cada uno: el formato está en
la [guía del Toolkit](../docs/guide/toolkit.md). El notebook se guarda sin resultados: lo que
veas al ejecutarlo serán tus datos.

In [ ]:
from pathlib import Path

from investinglab.config import LogLevel, configure_logging
from investinglab.domain import DriftAssumption, RebalancePolicy, SimulationConfig, SimulationModel
from investinglab.toolkit import Toolkit

configure_logging(LogLevel.WARNING)
tk = Toolkit.from_settings()
tk.providers

## 1. Descarga simultánea de N series (4.2)

Todas las series se piden a la vez. Si alguna falla, el lote no se interrumpe: el resultado
separa los éxitos de los fallos y explica la causa de cada uno.

In [ ]:
WEIGHTS = {"SAN.MC": 0.35, "ITX.MC": 0.25, "IBE.MC": 0.25, "BBVA.MC": 0.15}
START = "2019-01-01"

batch = tk.many(WEIGHTS, start=START)
batch.summary()

## 2. Series validadas con estadísticos automáticos (4.3)

Cada `PriceSeries` se valida al crearse (fechas ordenadas y sin duplicados, precios positivos,
divisa y procedencia) y calcula sus estadísticos en ese momento.

In [ ]:
san = batch.succeeded["SAN.MC"]
san

In [ ]:
san.stats

La calidad de los datos se comprueba con el calendario oficial del mercado. Las incidencias se
informan, nunca se borran.

In [ ]:
tk.quality(san).to_frame()

## 3. Cartera en euros con rebalanceo mensual

`portfolio()` descarga los activos, los convierte a la divisa base si hace falta y valida los
pesos.

In [ ]:
cartera = tk.portfolio(
    "Núcleo", WEIGHTS, base_currency="EUR", start=START, rebalance=RebalancePolicy.periodic("1mo")
)
metricas = cartera.metrics()
metricas.annualized_return, metricas.annualized_volatility, metricas.sharpe_ratio

## 4. Simulación de Monte Carlo (4.4)

Colas gruesas (t de Student) y sin apostar por la media histórica: deriva cero. La política de
rebalanceo de la cartera se aplica dentro de cada trayectoria.

In [ ]:
simulacion = cartera.monte_carlo(
    SimulationConfig(
        n_paths=10_000, horizon=252, model=SimulationModel.STUDENT_T, drift=DriftAssumption.ZERO
    )
)
simulacion.summary()

In [ ]:
simulacion.plot()

## 5. Informe y gráficos (4.5)

El informe usa el tipo libre de riesgo *point-in-time* de la divisa cuando hay una fuente que lo
sirva, y explica en «Diagnósticos y advertencias» hasta qué punto son fiables las cifras.

In [ ]:
informe = tk.report(cartera, simulation=simulacion)
informe

In [ ]:
graficos = cartera.plots_report()
graficos

In [ ]:
salida = Path("salidas")
informe.save(salida / "cartera.md")
tk.report(cartera, language="en", simulation=simulacion).save(salida / "portfolio.md")
graficos.save(salida / "graficos")
sorted(p.name for p in salida.rglob("*") if p.is_file())

In [ ]:
tk.close()